### Set working direcotry

In [ ]:
setwd("~/Projects/macrophage/")

# Library

In [ ]:
library(anndataR)
library(Seurat)
library(Matrix)
library(dplyr)
library(vsn)
library(DESeq2)
library(ggplot2)
library(reshape2)
library(pheatmap)
library(ggcorrplot)

# Load data

In [ ]:
load("./macrophage.RData")

# Create Seurat object using raw counts

In [ ]:
raw_data <- read_h5ad("./data/LungMAP_MouseLung_CellRef.v1.1.raw.h5ad")
raw_gene_names <- raw_data$var[["features"]]
head(raw_gene_names, 10)
raw_counts <- t(as.matrix(raw_data$X))
rownames(raw_counts) <- raw_gene_names


seurat_obj <- CreateSeuratObject(
  counts = raw_counts,
  meta.data = as.data.frame(raw_data$obs),
  min.features = 250,
  min.cells = 30
)

# Verify — this should now show real gene names
head(rownames(seurat_obj), 10)
dim(raw_counts)
length(raw_gene_names)


head(seurat_obj@assays[["RNA"]]@features)
seurat_obj@assays[["RNA"]]@features["Il34", ]


# Save Seurat object
SaveSeuratRds(object = seurat_obj, file = "./data/LungMAP_Seurat-raw_obj.rds")

# Normalize and scale data, PCA, UMAP for Seurat object 

In [ ]:
seurat.merged <- seurat_obj

## Analysis of cDNA assay only
DefaultAssay(seurat.merged) <- "RNA"

seurat.merged <- NormalizeData(seurat.merged)
seurat.merged <- FindVariableFeatures(seurat.merged)
seurat.merged <- ScaleData(seurat.merged)
seurat.merged <- RunPCA(seurat.merged, npcs = 50)
seurat.merged <- FindNeighbors(seurat.merged, dims = 1:30)
seurat.merged <- FindClusters(seurat.merged, resolution = 0.1, verbose = TRUE)
seurat.merged <- RunUMAP(seurat.merged, dims = 1:30)

options(repr.plot.width = 5, repr.plot.height = 5)
DimPlot(seurat.merged, label = TRUE, raster = TRUE) + NoLegend()

# Save Seurat.merged object to Rds
SaveSeuratRds(object = seurat.merged, file = "./data/LungMAP_Seurat-raw_merged.rds")

# Get AT2 cells from Seurat object

In [ ]:
# Subset Seurat object (merged) for AT2 cells
seurat_AT2 <- subset(
  seurat_obj,
  subset = celltype_level3_fullname == "Alveolar type 2 cell"
)

# AT2 Pseudobulk counts
at2_pseudo_counts <- AggregateExpression(
  object = seurat_AT2,
  group.by = "orig.ident",
  assays = "RNA",
  slot = "counts"
)
at2_matrix <- as.matrix(at2_pseudo_counts$RNA)

# Read in metadata .txt file
cell_meta <- read.table("data/Cell.Metadata.txt", header = TRUE, sep = "\t") %>% data.frame(stringsAsFactors = FALSE)
rownames(cell_meta) <- cell_meta$Cell.ID

# # Subset AT2 cell barcodes
# at2_cells <- cell_meta %>%
#   filter(CellType == "AT2") %>%
#  pull(Cell.ID) %>% data.frame
# rownames(at2_cells) <- at2_cells$.

# DESeq2

In [ ]:
# Read in metadata needed for DESeq2
deseq_metadata <- read.csv("./data/LungMAP-deseq2-metadata.csv")
deseq_metadata

at2_counts <- at2_matrix
head(at2_counts)

# Construct DESeq2 dataset object
at2_dds <- DESeqDataSetFromMatrix(
  countData = at2_counts,
  colData = deseq_metadata,
  design = ~age,
  tidy = FALSE # Would be TRUE if rownames were not already gene names and were instead present as the first column in the counts matrix
)

at2_dds <- DESeq(at2_dds)
# Save RDS of AT2 DESeq data set
saveRDS(at2_dds, "./data/AT2_DESeq2-dds.rds")

at2_results <- results(at2_dds, tidy = TRUE)
head(at2_results)

summary(at2_results)

# rlog-normalize AT2 cell expression counts

In [ ]:
at2_dds <- readRDS("./data/AT2_DESeq2-dds.rds")
# rlog transform the DESeq2 dataset: Remove the dependence of the variance on the mean, particularly the high variance of the logarithm of count data when the mean is low. There will be some elevated standard deviation in the lower count range
rld_at2 <- rlog(at2_dds, blind = FALSE) # blind=FALSE since design specifies 'age' as comparison group
head(assay(rld_at2)) # assay pulls out normalized values

# SD and PCA

In [ ]:
# Mean standard deviation plot
meanSdPlot(assay(rld_at2))

# PCA plot
rlog_at2_PCAData <- plotPCA(rld_at2, intgroup = "age", returnData = TRUE)

# Include percent of the variance explained by the top 2 principal components
percentVar <- round(100 * attr(rlog_at2_PCAData, "percentVar"), 2)

ggplot(rlog_at2_PCAData, aes(x = PC1, y = PC2, color = group)) +
  geom_point(size = 2) +
  geom_text(aes(label = name), vjust = 1.5, size = 3, show.legend = FALSE) +
  labs(
    title = "LungMAP PCA",
    x = paste0("PC1: ", percentVar[1], "%"),
    y = paste0("PC2: ", percentVar[2], "%")
  ) +
  theme_bw()

# AT2 cells Correlation (Spearman) calculation by timepoint (8 'samples')

In [ ]:
# Overall correlation global gene expression
rld_at2_matrix <- assay(rld_at2)
rld_at2_cor <- cor(rld_at2_matrix, method = "spearman")
pheatmap(rld_at2_cor)

In [ ]:
# rlog expression matrix for AT2 cells
expr <- rld_at2_matrix

target_gene <- "Il34"

# Extract target vector (as numeric)
target_vec <- as.numeric(expr[target_gene, ])
n_samples <- length(target_vec)

# Prepare output storage
genes <- rownames(expr)
n_genes <- length(genes)
results <- data.frame(gene = genes, rho = NA_real_, p.value = NA_real_, stringsAsFactors = FALSE)


# Loop through genes and compute Spearman correlation with Il34
for (i in seq_len(n_genes)) {
  g <- genes[i]
  # skip if same gene
  if (g == target_gene) {
    results$rho[i] <- 1
    results$p.value[i] <- 0
    next
  }
  # get numeric vector for gene g
  vec <- as.numeric(expr[g, ])
  # Check for enough non-NA pairs
  ok <- complete.cases(vec, target_vec)
  if (sum(ok) < 3) { # too few data points to compute correlation reliably
    results$rho[i] <- NA
    results$p.value[i] <- NA
    next
  }
  # compute Spearman correlation using cor.test
  ct <- suppressWarnings(cor.test(vec[ok], target_vec[ok], method = "spearman", exact = FALSE))
  results$rho[i] <- unname(ct$estimate) # Spearman's rho
  results$p.value[i] <- ct$p.value
}

# Post-process: remove rows with NA rho (if you want(optional), and sort by rho descending
# results <- results[!is.na(results$rho), ]
results <- results[order(-results$rho), ]

# Save to CSV
write.csv(results, "./data/AT2_Il34_spearman_by_gene.csv", row.names = FALSE)

## AT2 cells IL34 scatterplots vs every other gene (Spearman + best fit line)

In [ ]:
expr <- rld_at2_matrix
# Sample names
sample_names <- colnames(expr)
if (is.null(sample_names) || any(sample_names == "")) {
  sample_names <- paste0("Sample", seq_len(ncol(expr)))
}

# Output folder
plot_dir <- "./plots"
pdf_file <- file.path(plot_dir, "AT2_Il34_vs_all_genes_spearman_scatter.pdf")

# Genes to plot: all except Il34, and drop genes with NA rho from your results table
genes_to_plot <- results$gene[results$gene != target_gene]
genes_to_plot <- genes_to_plot[!is.na(results$rho[match(genes_to_plot, results$gene)])]

# Limit number of plots
max_plots <- 1000
genes_to_plot <- head(genes_to_plot, max_plots)

# Create a single multi-page PDF
pdf(pdf_file, width = 6, height = 5, onefile = TRUE)

for (g in genes_to_plot) {
  x <- as.numeric(expr[g, ])
  y <- target_vec

  df <- data.frame(
    sample = sample_names,
    other_gene = x,
    Il34 = y,
    stringsAsFactors = FALSE
  )
  df <- df[complete.cases(df$other_gene, df$Il34), , drop = FALSE]

  # Spearman correlation (for annotation)
  ct <- suppressWarnings(cor.test(df$other_gene, df$Il34, method = "spearman", exact = FALSE))
  rho <- unname(ct$estimate)
  pval <- ct$p.value

  # Linear model for the plotted best-fit line + r^2
  fit <- tryCatch(lm(Il34 ~ other_gene, data = df), error = function(e) NULL)
  r2 <- if (!is.null(fit)) summary(fit)$r.squared else NA_real_

  # Format stats
  rho_txt <- sprintf("%.3f", rho)
  p_txt <- format.pval(pval, digits = 3, eps = 1e-300)
  r2_txt <- if (is.na(r2)) "NA" else sprintf("%.3f", r2)

  # Scatter + best-fit line + Spearman + r^2
  p <- ggplot(df, aes(x = other_gene, y = Il34)) +
    geom_point(size = 2) +
    geom_smooth(method = "lm", se = FALSE) +
    # Label points (8 samples):
    geom_text(aes(label = sample), vjust = -0.7, size = 3) +
    labs(
      title = paste0("Il34 vs ", g),
      x = paste0(g, " (rlog expression)"),
      y = "Il34 (rlog expression)",
      subtitle = paste0(
        "Spearman rho = ", rho_txt,
        ", p = ", p_txt,
        " | r^2 ", r2_txt
      )
    ) +
    theme_bw()
}

dev.off()

# Cluster AT2 cell rlog(expression) matrix

In [ ]:
## Hierarchical clustering of genes (rows) in a genes x samples matrix
## - Produces: (1) CSV with gene -> cluster assignment, (2) PDF heatmap with dendrogram(s)

# -------------------- USER OPTIONS --------------------
expr <- rld_at2_matrix # <--- set this to your matrix object (genes=rows, samples=cols)
k <- 11 # number of gene clusters to cut the gene dendrogram into
linkage_method <- "ward.D2" # "complete", "average", "single", "ward.D2", ...
distance_metric <- "correlation" # "euclidean" or "correlation"
cor_method <- "spearman" # used only if distance_metric == "correlation": "pearson" or "spearman"

scale_rows <- TRUE # TRUE = z-score each gene across samples before clustering/plotting, so clustering REFLECTS EXPRESSION PATTERNS ACROSS SAMPLES, NOT EXPRESSION MAGNITUDE
cluster_samples <- FALSE # TRUE = also cluster samples (columns) and show sample dendrogram


# -------------------- PREP --------------------
mat <- as.matrix(expr)
stopifnot(!is.null(rownames(mat)), !is.null(colnames(mat)))

# Optional: row-scale (z-score) genes across samples
mat_use <- if (scale_rows) t(scale(t(mat))) else mat

# Drop genes with zero/undefined variance (helps correlation + scaling behave)
gene_sd <- apply(mat_use, 1, sd, na.rm = TRUE)
keep <- is.finite(gene_sd) & gene_sd > 0
mat_use <- mat_use[keep, , drop = FALSE]

if (k > nrow(mat_use)) stop("k is larger than the number of genes after filtering.")

# -------------------- GENE CLUSTERING --------------------
if (distance_metric == "euclidean") {
  d_genes <- dist(mat_use, method = "euclidean")
} else if (distance_metric == "correlation") {
  cor_genes <- cor(t(mat_use), method = cor_method, use = "pairwise.complete.obs")
  d_genes <- as.dist(1 - cor_genes)
} else {
  stop("distance_metric must be 'euclidean' or 'correlation'.")
}

hc_genes <- hclust(d_genes, method = linkage_method)
gene_clusters <- cutree(hc_genes, k = k)

# Save gene cluster assignments
cluster_df <- data.frame(gene = rownames(mat_use), cluster = gene_clusters)
write.csv(cluster_df, paste0("./data/AT2_gene_hclust", "_clusters_k", k, ".csv"), row.names = FALSE)

# -------------------- (OPTIONAL) SAMPLE CLUSTERING --------------------
hc_samples <- FALSE
if (cluster_samples) {
  if (distance_metric == "euclidean") {
    d_samples <- dist(t(mat_use), method = "euclidean")
  } else {
    cor_samples <- cor(mat_use, method = cor_method, use = "pairwise.complete.obs")
    d_samples <- as.dist(1 - cor_samples)
  }
  hc_samples <- hclust(d_samples, method = linkage_method)
}

# -------------------- HEATMAP WITH DENDROGRAM --------------------
annotation_row <- data.frame(Cluster = factor(gene_clusters))
rownames(annotation_row) <- rownames(mat_use)

clrsp <- colorRampPalette(c("blue", "white", "red"))
clrs <- clrsp(200)

pheatmap(
  mat_use,
  cluster_rows = hc_genes,
  cluster_cols = hc_samples, # FALSE if cluster_samples == FALSE
  cutree_rows = k, # visually separates the k gene clusters
  annotation_row = annotation_row,
  color = clrs,
  show_rownames = FALSE, # set TRUE if you have few genes and want names shown
  main = paste0(
    "Gene hclust (k=", k,
    ", linkage=", linkage_method,
    ", dist=", distance_metric,
    if (distance_metric == "correlation") paste0(" [", cor_method, "]") else "",
    if (scale_rows) ", row-zscore" else ""
  ),
  filename = paste0("./plots/AT2_gene_hclust", "_heatmap_k", k, ".pdf"),
  width = 7, height = 9
)

## Cluster analysis

In [ ]:
clusters <- read.table("./data/AT2_gene_hclust_clusters_k12.csv", sep = ",", header = TRUE)
cluster_6 <- subset(clusters, cluster == 6)
cluster6_genes <- cluster_6[, 1]
write.table(cluster6_genes, "./data/AT2_cluster6_genes.txt", row.names = FALSE, quote = FALSE, col.names = FALSE)